# Orca Core DPO — Merge + GGUF Export (standalone, decoupled from training)

**Why this notebook is separate from the training notebook**: the training
notebook (`orca_core_dpo_kaggle_v1.ipynb`) has now succeeded 4 times in a
row producing a real, valid LoRA adapter — but the merge+GGUF export tail
has failed 5 different ways in a row, all inside Unsloth's convenience
wrapper functions (`save_pretrained_merged` / `save_pretrained_gguf`).
Every one of those failures happened in the export step, which is pure
CPU work (tensor reshaping, dtype conversion, file I/O) — no GPU math
involved. Retrying the whole training notebook each time to debug an
export bug was burning scarce GPU-hour quota on a step that doesn't need
a GPU at all. This notebook fixes that by taking the adapter as a given
(already trained, already uploaded as its own dataset) and only doing the
merge + export, so iterating on this specific problem is cheap.

**Why this bypasses Unsloth's wrapper entirely**: the last failure proved
the problem isn't just a stale `quantization_config` tag in a JSON file —
even after stripping it and manually re-running the converter, the exact
same `NotImplementedError: Quant method is not yet supported: 'bitsandbytes'`
came back. That means the actual saved tensors were still genuinely in
packed 4-bit form, not truly dequantized fp16 — Unsloth's internal fusion
wasn't completing correctly for this specific combination. Rather than
keep patching around that wrapper, this notebook uses plain, standard
PEFT (`merge_and_unload()`) on a base model loaded directly in fp16 (never
4-bit in this notebook at all) — there's no quantization config anywhere
in this path, so there's nothing stale to strip and nothing to dequantize.

**What to attach as inputs before running**:
- Dataset: `orca-core-dpo-adapter-v1` (the trained adapter, already uploaded)

**GPU note**: this notebook still requests a T4 (loading an 8B model in
fp16 for the merge is tight on plain CPU RAM — 16GB+ just for weights).
But it's a short session (a merge + file conversion, not a training loop)
— expect a few minutes, not the 15-20 minute+ cost of a full retrain.

In [ ]:
!pip install -q "transformers<5" "torchao>=0.16.0" peft accelerate

## Find the uploaded adapter

In [ ]:
import glob, os

adapter_matches = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
print('Adapter config found at:', adapter_matches)

if not adapter_matches:
    raise FileNotFoundError(
        "Attach the orca-core-dpo-adapter-v1 dataset to this notebook before running."
    )

adapter_dir = os.path.dirname(adapter_matches[0])
print('adapter_dir =', adapter_dir)

## Load the base model in plain fp16 (never 4-bit) and merge the adapter

No `BitsAndBytesConfig` anywhere in this path -- `AutoModelForCausalLM`
loaded directly in fp16, so there is no quantization metadata for anything
downstream to trip over.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"

print("[load] loading base model in fp16...")
base = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
print("[load] base model loaded")

print("[merge] attaching adapter and merging...")
model = PeftModel.from_pretrained(base, adapter_dir)
merged = model.merge_and_unload()
print("[merge] done -- merged is a plain AutoModelForCausalLM, no PEFT wrapper, no quant config")

# Sanity check: confirm there is genuinely no quantization_config on this
# model before we spend time exporting it -- fail loudly here instead of
# discovering the same bug three levels deep in a subprocess again.
assert getattr(merged.config, "quantization_config", None) is None, (
    "merged model still has a quantization_config -- something upstream "
    "changed and this notebook's core assumption no longer holds, stop here."
)
print("[check] confirmed: no quantization_config on the merged model")

## Save the merged fp16 model (in /tmp, not /kaggle/working -- same 19.5GB quota reason as before)

In [ ]:
import shutil

shutil.rmtree("/tmp/merged_clean", ignore_errors=True)
merged.save_pretrained("/tmp/merged_clean", safe_serialization=True)
tokenizer.save_pretrained("/tmp/merged_clean")
print("[save] merged model saved to /tmp/merged_clean")
!ls -la /tmp/merged_clean

## Convert to GGUF using the standard llama.cpp project directly (not Unsloth's vendored copy)

Cloning the real llama.cpp repo instead of relying on Unsloth's internal
wrapper -- fewer layers of "magic" between us and the actual conversion
script, and it's the same tool everyone else uses for this, so any problem
here is a well-documented, googleable llama.cpp issue rather than an
Unsloth-specific one.

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
!pip install -q -r /tmp/llama.cpp/requirements.txt

In [ ]:
import os

os.makedirs("/tmp/gguf_out", exist_ok=True)
f16_path = "/tmp/gguf_out/orca-core-dpo.F16.gguf"

print("[convert] running llama.cpp's convert_hf_to_gguf.py...")
!python3 /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/merged_clean --outfile {f16_path} --outtype f16
print("[convert] done, checking output:")
!ls -la /tmp/gguf_out

## Build llama.cpp's quantize tool and produce the final Q4_K_M GGUF

This needs a real C++ build (cmake), which takes a couple of minutes --
expected, not a sign of a hang.

In [ ]:
!cmake -B /tmp/llama.cpp/build -S /tmp/llama.cpp -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=OFF
!cmake --build /tmp/llama.cpp/build --config Release -j --target llama-quantize

In [ ]:
import glob

quantize_bin_candidates = glob.glob("/tmp/llama.cpp/build/**/llama-quantize", recursive=True)
print("[quantize] llama-quantize binary found at:", quantize_bin_candidates)

if not quantize_bin_candidates:
    raise RuntimeError(
        "llama-quantize binary not found after build -- check the cmake build "
        "output above for the real error before assuming this notebook's logic "
        "is wrong; this is a standard llama.cpp build step, not custom code."
    )

q4_path = "/tmp/gguf_out/orca-core-dpo.Q4_K_M.gguf"
!{quantize_bin_candidates[0]} {f16_path} {q4_path} Q4_K_M
print("[quantize] done, checking output:")
!ls -la /tmp/gguf_out

## Copy the final Q4_K_M GGUF to /kaggle/working (only the final file -- the F16 intermediate stays in /tmp, it's too big for the 19.5GB working quota)

In [ ]:
import shutil

dest = "/kaggle/working/orca-core-dpo.Q4_K_M.gguf"
shutil.copy(q4_path, dest)
print(f"[export] copied final quantized model to {dest}")
print("\nNext: click 'Save Version' -> 'Save & Run All (Commit)' at the top right.")